# Module 1 · Embeddings — In-class lab 🧪
## Mini-lab D — Document retrieval encoders under interrogation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mago-cinv/course-template/blob/master/modules/01-embeddings/labs/in-class/lab-d-document-retrieval.ipynb)

This mini-lab pairs with **Lesson 2** (pooling, RNNs, the vanishing gradient, LSTM gating) and **Lesson 3** (cosine similarity, LSH). Every claim below cites the lesson and equation it comes from — go back to the deck whenever a formula needs refreshing.

In mini-labs A–C you made *word* vectors. Here you get **document** vectors — three encoders, already built and trained for you — and your job is to **interrogate them**.

### How this mini-lab works

**The machinery is given to you.** The corpus, the vocabulary, the GloVe matrix, the mean-pooling baseline, the RNN and LSTM encoders, the training loop, the retrieval scorer, the LSH index and the plotting helpers are all written, commented and *already run* below. You are *not* asked to implement `nn.LSTM` or look up what `topk` returns — that is plumbing, and plumbing is not the lesson.

**What you write is the investigation.** Sections 🔬 **E7**–**E9** hand you the same toolkit and ask you to *use* it: turn a knob and measure what changes, build a comparison the notebook didn't make, and defend a short conclusion with numbers you produced. There is no single right answer — there is a defensible one.

Treat everything above the 🔬 sections as your **sandbox**: read it, run it, then take it apart.

| Mini-lab D asks | Section |
|---|---|
| Does word order actually matter for this task? | E7 |
| Does the RNN↔LSTM gap grow with sequence length? | E8 |
| What does the LSH speed/recall trade-off really cost? | E9 |

The code is a few lines of glue — it is not the point of the exercise, but it does need to **run**, because your conclusions have to cite numbers *you* produced. Each experiment ends with a `✓` check so you know your run is in good shape, and each conclusion is **3–4 sentences** — same as mini-labs A–C.

Everything runs on CPU in roughly **1–2 minutes**.


In [ ]:
# Setup — run me first (works on Colab AND locally)
# On Colab this installs the few extra libraries; locally the course venv
# (.venv, environment/requirements.txt) already has everything.
import sys

if "google.colab" in sys.modules:
    %pip install -q datasets gensim

print("Setup OK — running on", "Colab" if "google.colab" in sys.modules else "local Python")

In [ ]:
# Imports + seeds — one reseed helper, called before EVERY training run so comparisons are fair
import random
import string
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn


def reseed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


reseed(0)
print("torch", torch.__version__)

## 1 · The data (provided)

**DBpedia-14** — Wikipedia abstracts in **14 fine-grained classes** (Company, Artist, Athlete, Village, Album, Film, …). Two deterministic, **disjoint** subsets carved from a shuffled slice:

- **gallery** = **20,000** abstracts — the search index *and* the encoder training set;
- **queries** = **500** more — held out, never trained on.

The class label is our **relevance signal**: a retrieved neighbour counts as relevant iff it shares the query's class. With 14 classes, **chance precision@10 is 0.071** — a far more demanding floor than a 4-class problem, and the reason word *order* finally has something to contribute here.

In [ ]:
# Load DBpedia-14 and carve the two disjoint subsets
from datasets import load_dataset

# ⚠️ DBpedia ships sorted BY CLASS, so we must shuffle before slicing — otherwise the
# gallery would contain only the first few categories.
dbp = load_dataset("fancyzhx/dbpedia_14", split="train").shuffle(seed=0)
gallery_ds, queries_ds = dbp.select(range(20000)), dbp.select(range(20000, 20500))

gallery_texts = gallery_ds["content"]
gallery_labels = torch.tensor(gallery_ds["label"])
query_texts = queries_ds["content"]
query_labels = torch.tensor(queries_ds["label"])

CLASS_NAMES = ["Company", "School", "Artist", "Athlete", "Politician", "Transport",
               "Building", "NaturalPlace", "Village", "Animal", "Plant", "Album",
               "Film", "WrittenWork"]
CHANCE = 1.0 / len(CLASS_NAMES)
print(f"gallery: {len(gallery_texts):,} docs | queries: {len(query_texts)} docs (disjoint)")
print(f"{len(CLASS_NAMES)} classes -> chance precision@10 = {CHANCE:.3f}")
print("gallery class counts:", torch.bincount(gallery_labels).tolist())
print(f"\nexample: {gallery_texts[0][:110].strip()}…  ->  {CLASS_NAMES[gallery_labels[0]]}")

In [ ]:
# Pretrained GloVe (50-d, 400k words) — ~66 MB the first time, cached afterwards
import gensim.downloader

glove = gensim.downloader.load("glove-wiki-gigaword-50")
print(f"GloVe: {len(glove.key_to_index):,} words, dim {glove.vector_size}")

## 2 · Your toolkit (provided)

Everything below is written and working. **This is the sandbox** — E7–E9 are built entirely from these.

| Tool | What it gives you |
|---|---|
| `rebuild_ids(max_len)` | re-tokenize both splits at any sequence length → `(gallery_ids, query_ids)` |
| `mean_pool(ids)` | the order-blind baseline encoder (50-d) |
| `make_encoder(kind)` | a fresh `"rnn"` / `"lstm"` encoder, GloVe-warm-started |
| `train_encoder(model, ids, labels, epochs)` | shared training loop — same budget for every encoder |
| `encode_all(model, ids)` | document embeddings $h_T$, in `eval()` mode under `no_grad()` |
| `precision_at_k(q, g, ql, gl, k)` | the retrieval metric: mean fraction of top-$k$ sharing the query's class |
| `shuffle_tokens(ids, seed)` | randomly permute each document's real tokens (**E7's knob**) |
| `build_lsh(embs, K, L, seed)` / `lsh_eval(...)` | random-hyperplane index + (recall@10, avg candidates) (**E9's knobs**) |
| `show_retrieval(qi, ...)` | print a query and its top-$k$ neighbours as **text** (**E9's word-level view**) |

Every knob is named and defaulted, so you can change one without reading any library documentation.

In [ ]:
# ══ THE TOOLKIT (part 1: data → tensors) ══ provided; read it, then use it


def tokenize(text):
    """Lowercase, split on whitespace, strip punctuation stuck to token edges."""
    return [w for w in (t.strip(string.punctuation) for t in text.lower().split()) if w]


# Vocabulary = (corpus ∩ GloVe), capped at V, with [PAD]=0 and [UNK]=1
V, PAD_ID, UNK_ID = 20000, 0, 1
_counts = Counter(w for t in gallery_texts for w in tokenize(t))
itos = ["[PAD]", "[UNK]"] + [w for w, _ in _counts.most_common() if w in glove.key_to_index][: V - 2]
stoi = {w: i for i, w in enumerate(itos)}


def numericalize(texts, max_len):
    """(N, max_len) LongTensor of ids, PAD-padded / truncated."""
    out = torch.full((len(texts), max_len), PAD_ID, dtype=torch.long)
    for i, t in enumerate(texts):
        ids = [stoi.get(w, UNK_ID) for w in tokenize(t)][:max_len]
        out[i, : len(ids)] = torch.tensor(ids)
    return out


def rebuild_ids(max_len):
    """Both splits re-tokenized at a new sequence length — E8's knob."""
    return numericalize(gallery_texts, max_len), numericalize(query_texts, max_len)


# The GloVe embedding matrix: row i = vector of itos[i];  [PAD] = zeros, [UNK]/missing = mean vector
_mean_vec = torch.tensor(glove.vectors.mean(axis=0))
embed_matrix = torch.zeros(len(itos), glove.vector_size)
for _i, _w in enumerate(itos):
    if _w == "[PAD]":
        continue
    embed_matrix[_i] = torch.tensor(glove[_w]) if _w in glove.key_to_index else _mean_vec

MAX_LEN = 40
gallery_ids, query_ids = rebuild_ids(MAX_LEN)
print(f"vocab {len(itos)} | embed_matrix {tuple(embed_matrix.shape)} | "
      f"gallery_ids {tuple(gallery_ids.shape)} | query_ids {tuple(query_ids.shape)}")

In [ ]:
# ══ THE TOOLKIT (part 2: encoders, training, retrieval) ══ provided


def mean_pool(ids):
    """The order-blind baseline: average the GloVe rows of each doc's non-[PAD] tokens.
    Exactly invariant to token order — which is precisely what E7 exploits."""
    mask = (ids != PAD_ID).float()
    summed = (embed_matrix[ids] * mask.unsqueeze(-1)).sum(dim=1)
    return summed / mask.sum(dim=1, keepdim=True).clamp_min(1.0)


class RecurrentEncoder(nn.Module):
    """GloVe-warm-started embedding → recurrent cell → h_T is the document embedding → linear head.
    forward(ids) -> (logits (B,4), doc_embedding (B,hidden))."""

    def __init__(self, kind="lstm", hidden=64, n_classes=len(CLASS_NAMES)):
        super().__init__()
        n, d = embed_matrix.shape
        self.embedding = nn.Embedding(n, d, padding_idx=PAD_ID)
        self.embedding.weight.data.copy_(embed_matrix)      # warm start, stays trainable
        self.rnn = {
            "rnn": lambda: nn.RNN(d, hidden, nonlinearity="tanh", batch_first=True),
            "lstm": lambda: nn.LSTM(d, hidden, batch_first=True),
        }[kind]()
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, ids):
        out = self.rnn(self.embedding(ids))
        h = out[1][0] if isinstance(self.rnn, nn.LSTM) else out[1]   # LSTM returns (h, c)
        doc = h[-1]                                                  # (B, hidden) = h_T
        return self.head(doc), doc


def make_encoder(kind="lstm", seed=0):
    """A fresh encoder with reproducible weights. kind in {'rnn', 'lstm'}."""
    reseed(seed)
    return RecurrentEncoder(kind)


# 6 epochs: at 4 the LSTM merely TIES the mean-pool baseline (~0.80 either way);
# by 6 it has pulled clearly ahead (~0.82) while the whole run still finishes in
# ~15s on CPU for both encoders combined.
EPOCHS, BATCH_SIZE, LR = 6, 64, 1e-3


def train_encoder(model, ids, labels, epochs=EPOCHS, verbose=True):
    """Cross-entropy training. Identical budget + batch order for every encoder, so
    any difference you measure comes from the CELL, not from luck."""
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    shuffler = torch.Generator().manual_seed(0)
    history = {"loss": [], "acc": []}
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(len(ids), generator=shuffler)
        tot_loss = tot_ok = 0
        for i in range(0, len(ids), BATCH_SIZE):
            b = perm[i : i + BATCH_SIZE]
            logits, _ = model(ids[b])
            loss = loss_fn(logits, labels[b])
            opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * len(b)
            tot_ok += (logits.argmax(1) == labels[b]).sum().item()
        history["loss"].append(tot_loss / len(ids))
        history["acc"].append(tot_ok / len(ids))
        if verbose:
            print(f"    epoch {epoch+1}/{epochs}: loss {history['loss'][-1]:.3f}  "
                  f"acc {history['acc'][-1]:.3f}")
    return history


def encode_all(model, ids, batch_size=256):
    """Document embeddings for every row — eval mode, no gradients (Module 0 hygiene)."""
    model.eval()
    with torch.no_grad():
        return torch.cat([model(ids[i : i + batch_size])[1] for i in range(0, len(ids), batch_size)])


def precision_at_k(query_embs, gallery_embs, q_labels, g_labels, k=10):
    """PRECISION@k — the metric every number in this notebook is reported in.

    For each query: rank the whole gallery by COSINE similarity, take the k nearest,
    and compute the fraction of those k that share the query's class label. Then
    average that fraction over all queries. Chance = 1/n_classes = 0.071 here (14 classes).
    """
    q = query_embs / query_embs.norm(dim=1, keepdim=True).clamp_min(1e-8)
    g = gallery_embs / gallery_embs.norm(dim=1, keepdim=True).clamp_min(1e-8)
    topk = (q @ g.T).topk(k, dim=1).indices
    return (g_labels[topk] == q_labels[:, None]).float().mean().item()


def shuffle_tokens(ids, seed=0):
    """Randomly permute each document's REAL tokens, leaving [PAD] in place — E7's knob.
    Destroys word order while keeping the exact bag of words."""
    g = torch.Generator().manual_seed(seed)
    out = ids.clone()
    for i in range(len(ids)):
        n = int((ids[i] != PAD_ID).sum())
        if n > 1:
            out[i, :n] = ids[i, :n][torch.randperm(n, generator=g)]
    return out


def cos_rowwise(a, b):
    """Cosine similarity between MATCHING rows of a and b — a[i] vs b[i], not all-pairs.
    E7's second instrument: how far does a document's own embedding move when you
    shuffle its words? 1.000 = the encoder did not notice the shuffle at all."""
    an = a / a.norm(dim=1, keepdim=True).clamp_min(1e-8)
    bn = b / b.norm(dim=1, keepdim=True).clamp_min(1e-8)
    return (an * bn).sum(1)


print("toolkit part 2 ready")

In [ ]:
# ══ THE TOOLKIT (part 3: LSH, inspection, plotting) ══ provided


def build_lsh(embs, K=12, L=3, seed=42):
    """Random-hyperplane LSH: L independent tables of K hyperplanes each.
    bit_k(x) = 1[r_k . x >= 0]; collision probability is a monotone function of cosine
    similarity, so similar vectors land in the same bucket ON PURPOSE. E9's knobs are K and L."""
    rng = np.random.default_rng(seed)
    weights = 2 ** torch.arange(K)
    tables = []
    for _ in range(L):
        planes = torch.tensor(rng.standard_normal((K, embs.shape[1])), dtype=torch.float32)
        keys = ((embs @ planes.T >= 0).long() * weights).sum(1)
        buckets = defaultdict(list)
        for idx, key in enumerate(keys.tolist()):
            buckets[key].append(idx)
        tables.append((planes, buckets, weights))
    return tables


def lsh_eval(tables, query_embs, gallery_embs, k=10):
    """(recall@10 vs brute force, avg candidate-set size) for the given index."""
    qn = query_embs / query_embs.norm(dim=1, keepdim=True).clamp_min(1e-8)
    gn = gallery_embs / gallery_embs.norm(dim=1, keepdim=True).clamp_min(1e-8)
    exact = (qn @ gn.T).topk(k, dim=1).indices
    recalls, sizes = [], []
    for qi in range(len(query_embs)):
        cands = set()
        for planes, buckets, weights in tables:
            key = int(((query_embs[qi : qi + 1] @ planes.T >= 0).long() * weights).sum())
            cands.update(buckets.get(key, []))
        sizes.append(len(cands))
        if not cands:
            recalls.append(0.0)
            continue
        cl = sorted(cands)
        top = (qn[qi : qi + 1] @ gn[cl].T).topk(min(k, len(cl)), dim=1).indices[0]
        recalls.append(len({cl[j] for j in top.tolist()} & set(exact[qi].tolist())) / k)
    return float(np.mean(recalls)), float(np.mean(sizes))


def show_retrieval(qi, query_embs, gallery_embs, k=5, chars=95):
    """Print query qi and its top-k neighbours AS TEXT, with labels and match flags.
    precision@k counts a neighbour correct when the LABEL matches —
    this lets you read whether it is actually ABOUT the same thing."""
    q = query_embs / query_embs.norm(dim=1, keepdim=True).clamp_min(1e-8)
    g = gallery_embs / gallery_embs.norm(dim=1, keepdim=True).clamp_min(1e-8)
    top = (q[qi : qi + 1] @ g.T).topk(k, dim=1)
    print(f"QUERY [{CLASS_NAMES[query_labels[qi]]}]  {query_texts[qi][:chars]}…")
    for rank, (sim, gi) in enumerate(zip(top.values[0].tolist(), top.indices[0].tolist()), 1):
        same = gallery_labels[gi] == query_labels[qi]
        print(f"  {rank}. [{'MATCH ' if same else 'no    '}] "
              f"{CLASS_NAMES[gallery_labels[gi]]:<9} cos={sim:.3f}  {gallery_texts[gi][:chars]}…")


def lsh_retrieve(tables, query_embs, gallery_embs, k=10):
    """Top-k gallery indices for every query, searching ONLY the LSH candidate set.
    Returns (topk_indices, avg_candidates). Rows are padded with -1 if a bucket is short.
    E9's instrument for asking whether the cheap search still returns the same TOPICS."""
    qn = query_embs / query_embs.norm(dim=1, keepdim=True).clamp_min(1e-8)
    gn = gallery_embs / gallery_embs.norm(dim=1, keepdim=True).clamp_min(1e-8)
    out, sizes = [], []
    for qi in range(len(query_embs)):
        cands = set()
        for planes, buckets, weights in tables:
            key = int(((query_embs[qi : qi + 1] @ planes.T >= 0).long() * weights).sum())
            cands.update(buckets.get(key, []))
        sizes.append(len(cands))
        cl = sorted(cands)
        if not cl:
            out.append([-1] * k)
            continue
        top = (qn[qi : qi + 1] @ gn[cl].T).topk(min(k, len(cl)), dim=1).indices[0].tolist()
        idx = [cl[j] for j in top]
        out.append(idx + [-1] * (k - len(idx)))
    return torch.tensor(out), float(np.mean(sizes))


def topic_precision(topk, q_labels, g_labels, k=10):
    """precision@k computed from an ALREADY-RETRIEVED top-k index matrix (ignores -1 padding).
    Lets you score brute-force and LSH results on exactly the same footing."""
    total = 0.0
    for i, row in enumerate(topk):
        valid = row[row >= 0]
        if len(valid):
            total += (g_labels[valid] == q_labels[i]).float().sum().item() / k
    return total / len(topk)


print("toolkit part 3 ready — sandbox complete")

## 3 · Worked example — the three baselines

Three ways to turn a document into a vector, all trained on the same data with the same budget and the same seed — only the encoder changes:

1. **mean-pool** — average the GloVe vectors, ignore order entirely (**Lesson 2**, eq. 5);
2. **vanilla RNN** — order-aware, but its gradient is a product of ~40 Jacobians (**Lesson 2**, eq. 9–10);
3. **LSTM** — order-aware *with* the additive cell-state highway $\partial c_t/\partial c_{t-1} = \mathrm{diag}(\Gamma_f)$ (**Lesson 2**, eq. 13).

All three are scored by **precision@10**, built on the cosine similarity of **Lesson 3**, eq. 1: for each query, take its 10 nearest gallery documents by cosine similarity, measure what fraction share the query's class label, and average that over all queries. Chance ≈ 0.071 (14 classes).

*Expected:* mean-pooling is a surprisingly strong baseline; the LSTM edges past it; the vanilla RNN falls far behind. These numbers are your **reference point** — every experiment below is a deviation from this table.

In [ ]:
# Train all three and score them — this is the table E7–E9 will perturb
BASE = {}

BASE["mean-pool"] = precision_at_k(mean_pool(query_ids), mean_pool(gallery_ids),
                                   query_labels, gallery_labels)
print(f"mean-pool     p@10 = {BASE['mean-pool']:.3f}   (no training needed)")

encoders, histories = {}, {}
for kind in ["rnn", "lstm"]:
    print(f"\n{kind.upper()} encoder:")
    enc = make_encoder(kind)
    histories[kind] = train_encoder(enc, gallery_ids, gallery_labels)
    encoders[kind] = enc
    BASE[kind] = precision_at_k(encode_all(enc, query_ids), encode_all(enc, gallery_ids),
                                query_labels, gallery_labels)

print("\n" + "=" * 46)
print(f"{'encoder':<14}{'p@10':>8}{'train acc':>12}")
for name in ["mean-pool", "rnn", "lstm"]:
    acc = f"{histories[name]['acc'][-1]:.3f}" if name in histories else "—"
    print(f"{name:<14}{BASE[name]:>8.3f}{acc:>12}")
print("=" * 46)

# Keep the trained LSTM's embeddings around — E9 uses them
gallery_lstm = encode_all(encoders["lstm"], gallery_ids)
query_lstm = encode_all(encoders["lstm"], query_ids)

## 4 · Worked example — what retrieval actually looks like

Before you trust a number, look at what it is summarising. `show_retrieval` prints a query and its nearest neighbours as **text**, flagging which ones `precision@10` counted as correct.

*Expected:* the flagged `MATCH`es all share the query's class — a first look at what a "nearest neighbour" actually is, in words rather than in a number.

In [ ]:
show_retrieval(0, query_lstm, gallery_lstm, k=5)

---

## 🔬 E7 — Does word order actually matter?

The LSTM beats mean-pooling by only a hair. The standard story is "order-aware encoders are better" — but if that were the whole story the gap should be *large*, because mean-pooling throws order away completely.

**So test whether order is worth anything on this task at all.** `shuffle_tokens(ids)` randomly permutes each document's real tokens: the bag of words is untouched, the order is destroyed.

**Measure it at two levels — that contrast is the whole point.**

1. **Retrieval level.** Score **all three encoders** on the shuffled input, **without retraining**, and store `order_ablation[name] = {"normal": …, "shuffled": …, "drop": normal - shuffled}`. Plot the three normal-vs-shuffled pairs as a grouped bar chart.
2. **Embedding level.** For each encoder, compare **each document's own embedding** before and after shuffling with `cos_rowwise` — store the mean in the same dict as `"cos_self"`. This asks a stricter question: *did the encoder even notice?* A drop in p@10 can be hidden by the task; a moved embedding cannot.

**Predict before you run it:** which encoder must score **exactly** `cos_self = 1.000`, and why? Your conclusion must state that prediction and whether it held.

> Everything you need: `shuffle_tokens`, `mean_pool`, `encode_all`, `precision_at_k`, `cos_rowwise`, and the trained models in `encoders`.

In [ ]:
# TODO: build shuffled ids for both splits with shuffle_tokens(...)
# TODO: score all three encoders on shuffled input WITHOUT retraining -> `order_ablation`
# TODO: also record "cos_self" per encoder with cos_rowwise (embedding-level test)
# TODO: plot normal vs shuffled as grouped bars
# HINT: sh_gallery, sh_query = shuffle_tokens(gallery_ids), shuffle_tokens(query_ids)
# HINT: cos_rowwise(embs_normal, embs_shuffled).mean() -> one number per encoder

sh_gallery, sh_query = shuffle_tokens(gallery_ids), shuffle_tokens(query_ids)

order_ablation = {}
for name in ["mean-pool", "rnn", "lstm"]:
    if name == "mean-pool":
        qn, gn = mean_pool(query_ids), mean_pool(gallery_ids)
        qs, gs = mean_pool(sh_query), mean_pool(sh_gallery)
    else:
        enc = encoders[name]
        qn, gn = encode_all(enc, query_ids), encode_all(enc, gallery_ids)
        qs, gs = encode_all(enc, sh_query), encode_all(enc, sh_gallery)
    normal = precision_at_k(qn, gn, query_labels, gallery_labels)
    shuffled = precision_at_k(qs, gs, query_labels, gallery_labels)
    order_ablation[name] = {
        "normal": normal,
        "shuffled": shuffled,
        "drop": normal - shuffled,
        "cos_self": float(cos_rowwise(qn, qs).mean()),   # embedding-level test
    }

print(f"{'encoder':<12}{'normal':>9}{'shuffled':>10}{'drop':>8}{'cos_self':>11}")
for name, r in order_ablation.items():
    print(f"{name:<12}{r['normal']:>9.3f}{r['shuffled']:>10.3f}{r['drop']:>+8.3f}{r['cos_self']:>11.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
names = list(order_ablation)
x = np.arange(len(names))
ax[0].bar(x - 0.2, [order_ablation[n]["normal"] for n in names], 0.4, label="normal")
ax[0].bar(x + 0.2, [order_ablation[n]["shuffled"] for n in names], 0.4, label="shuffled")
ax[0].axhline(CHANCE, ls="--", c="gray", lw=1, label="chance")
ax[0].set_xticks(x); ax[0].set_xticklabels(names)
ax[0].set(ylabel="precision@10", title="Retrieval level: barely moves")
ax[0].legend(fontsize=8)
ax[1].bar(x, [order_ablation[n]["cos_self"] for n in names], 0.5, color="crimson")
ax[1].axhline(1.0, ls="--", c="gray", lw=1)
ax[1].set_xticks(x); ax[1].set_xticklabels(names)
ax[1].set(ylabel="cos(original, shuffled)", ylim=(0.8, 1.02),
          title="Embedding level: the encoder DID notice")
plt.tight_layout(); plt.show()

In [ ]:
# Light check on YOUR experiment (your numbers are your own — this only checks the shape)
try:
    assert set(order_ablation) == {"mean-pool", "rnn", "lstm"}, \
        f"order_ablation needs the keys mean-pool/rnn/lstm, got {set(order_ablation)}"
    assert all({"normal", "shuffled", "drop", "cos_self"} <= set(r) for r in order_ablation.values()), \
        "each entry needs 'normal', 'shuffled', 'drop' and 'cos_self'"
    assert order_ablation["mean-pool"]["cos_self"] > 0.9999, \
        "mean-pooling is an average, so shuffling MUST leave its embedding in place " \
        "(cos_self = 1.000). Check your shuffle."
    assert abs(order_ablation["mean-pool"]["drop"]) < 0.005, \
        "mean-pooling's p@10 should barely move under shuffling. Check your shuffle."
    print("✓ E7 experiment complete")
    for name, r in order_ablation.items():
        print(f"   {name:<11} {r['normal']:.3f} → {r['shuffled']:.3f}  "
              f"(drop {r['drop']:+.3f})   cos_self {r['cos_self']:.4f}")
except NameError:
    print("✗ E7 not attempted yet — write the experiment in the cell above, then re-run.")
except Exception as e:
    print(f"✗ E7 check failed: {e}")

### ✍️ E7 conclusion


> TODO: your conclusion here (3–4 sentences)

**Model answer.**

**The prediction held: mean-pooling scored `cos_self` = 1.0000.** An average is permutation-invariant by construction, so shuffling cannot move its embedding — the `✓` check enforces exactly this, and its p@10 is unchanged at **0.906**.

**This time word order genuinely pays.** The LSTM falls **0.966 → 0.903** when its input is reduced to a bag of words — a drop of **+0.063 (6.5%)**, against a chance floor of just 0.071. That is a real, large effect: roughly a *quarter* of the LSTM's advantage over mean-pooling (0.966 − 0.906 = 0.060) evaporates the moment order is destroyed.

**Why here and not on a 4-class topic task?** With **14 fine-grained classes**, the label is no longer decidable from topic vocabulary alone. Telling a *Village* from a *Building*, or an *Album* from a *WrittenWork*, turns on how the abstract is *phrased* — "X **is a** village **in** Y" versus "X **was a** building **designed by** Y" share most of their content words and differ in structure. That is exactly the signal a bag of words throws away and a recurrent encoder can keep.

**The embedding level confirms the mechanism.** LSTM `cos_self` = **0.9526**, RNN **0.8836** — both encoders' document vectors visibly *move* under shuffling, and this time the move is large enough to change which class they land nearest. (The RNN is a special case: at **0.285** it barely learned the task at all, so its −0.011 "improvement" under shuffling is noise around a near-chance encoder, not evidence about order.)

---

## 🔬 E8 — Does the RNN↔LSTM gap grow with sequence length?

**Lesson 2** (eq. 9–10) makes a **quantitative** claim, not a vague one. The vanilla RNN's gradient from the loss back to token $k$ is a product of $T-k$ Jacobians:

$$\frac{\partial \mathcal{L}_T}{\partial h_k} = \frac{\partial \mathcal{L}_T}{\partial h_T}\prod_{t=k+1}^{T} \mathrm{diag}(g')\,W_{hh}, \qquad \lVert\cdot\rVert \lesssim r^{T-k}$$

If that exponential decay is real, then **the RNN's disadvantage must get worse as documents get longer** — and at very short lengths the two cells should be nearly indistinguishable, because there is barely any distance for the gradient to travel.

**That is a falsifiable prediction. Test it.**

**Design and run the experiment:**

1. For `max_len ∈ [10, 20, 30, 40]`: `rebuild_ids(max_len)`, then train **a fresh RNN and a fresh LSTM** on it (use `epochs=4` to keep the sweep quick) and score both with `precision_at_k`.
   *(Why stop at 40? DBpedia abstracts average **46.1 real tokens**. Past ~40 you are not testing longer documents, you are testing longer `[PAD]` runs — at `max_len=80` **52% of every sequence is padding** and both encoders collapse to chance. That is a property of the padding, not of recurrence.)*
2. Store your measurements in a dict called `length_sweep` mapping each `max_len` to `{"rnn": ..., "lstm": ..., "gap": lstm - rnn}`.
3. Plot both curves against sequence length on one axis, and the gap on a second panel.

⚠️ This is the slowest cell in the notebook (~1 min) and it runs silently. Train a **fresh** encoder at every length — reusing a model trained at a different length would invalidate the comparison.

> Everything you need: `rebuild_ids`, `make_encoder`, `train_encoder`, `encode_all`, `precision_at_k`.

In [ ]:
# TODO: for each max_len, rebuild ids and train a FRESH rnn and lstm on it
# TODO: build `length_sweep` = {max_len: {"rnn":…, "lstm":…, "gap":…}, …}
# TODO: print a row per length, then plot both curves vs length plus the gap
# HINT: g_ids, q_ids = rebuild_ids(L)
# HINT: enc = make_encoder(kind); train_encoder(enc, g_ids, gallery_labels, epochs=4, verbose=False)

length_sweep = {}
for L in [10, 20, 30, 40]:
    g_ids, q_ids = rebuild_ids(L)
    row = {}
    for kind in ["rnn", "lstm"]:
        enc = make_encoder(kind)
        train_encoder(enc, g_ids, gallery_labels, epochs=4, verbose=False)
        row[kind] = precision_at_k(encode_all(enc, q_ids), encode_all(enc, g_ids),
                                   query_labels, gallery_labels)
    row["gap"] = row["lstm"] - row["rnn"]
    length_sweep[L] = row
    print(f"max_len {L:3d}:  RNN {row['rnn']:.3f}   LSTM {row['lstm']:.3f}   gap {row['gap']:+.3f}")

lens = list(length_sweep)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(lens, [length_sweep[L]["lstm"] for L in lens], marker="o", label="LSTM")
ax[0].plot(lens, [length_sweep[L]["rnn"] for L in lens], marker="s", label="vanilla RNN")
ax[0].axhline(CHANCE, ls="--", c="gray", lw=1, label="chance")
ax[0].set(xlabel="sequence length (tokens)", ylabel="precision@10", title="Retrieval vs. length")
ax[1].plot(lens, [length_sweep[L]["gap"] for L in lens], marker="D", c="crimson", label="gap")
ax[1].axhline(0, ls="--", c="gray", lw=1)
ax[1].set(xlabel="sequence length (tokens)", ylabel="p@10(LSTM) − p@10(RNN)",
          title="Does the gap grow with distance?")
for a in ax:
    a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Light check on YOUR experiment (structure only)
try:
    assert len(length_sweep) >= 3, "sweep at least 3 sequence lengths so a trend is visible"
    assert all({"rnn", "lstm", "gap"} <= set(r) for r in length_sweep.values()), \
        "each entry needs 'rnn', 'lstm' and 'gap'"
    assert all(abs(r["gap"] - (r["lstm"] - r["rnn"])) < 1e-9 for r in length_sweep.values()), \
        "'gap' must equal lstm - rnn"
    lo, hi = min(length_sweep), max(length_sweep)
    print("✓ E8 experiment complete")
    print(f"   lengths tested : {sorted(length_sweep)}")
    for L in sorted(length_sweep):
        r = length_sweep[L]
        print(f"   len {L:<4} RNN {r['rnn']:.3f}  LSTM {r['lstm']:.3f}  gap {r['gap']:+.3f}")
    print(f"   → from len {lo} to len {hi} the gap went {length_sweep[lo]['gap']:+.3f} → "
          f"{length_sweep[hi]['gap']:+.3f}")
except NameError:
    print("✗ E8 not attempted yet — write the experiment in the cell above, then re-run.")
except Exception as e:
    print(f"✗ E8 check failed — {type(e).__name__}: {e}")


### ✍️ E8 conclusion


> TODO: your conclusion here (3–4 sentences)

**Model answer.**

**The gap grows monotonically with sequence length, exactly as Lesson 2 predicts.** Gaps: **+0.031** (len 10) → **+0.056** (len 20) → **+0.321** (len 30) → **+0.659** (len 40). At 10 tokens the two cells are close (RNN 0.826 vs LSTM 0.857) — there is barely any distance for the gradient to cross, so gating has little to protect.

**Only one of the two curves moves.** The LSTM *improves* and then holds (0.857 → 0.959 → 0.973 → **0.956**), while the RNN rises and then **collapses** (0.826 → 0.903 → 0.653 → **0.297**, against a 0.071 floor). The asymmetry is the whole point: eq. (9)–(10)'s exponential decay is a claim about the *vanilla* recurrence, and eq. (13)'s additive cell path is what exempts the LSTM from it.

**Read the mechanism, not just the curve.** The RNN's gradient back to token $k$ carries $T-k$ copies of $\mathrm{diag}(g')W_{hh}$, so by length 40 the earliest tokens receive almost no learning signal — and in a DBpedia abstract the opening clause ("X is a village in…") is precisely where the class is decided. The LSTM's $\partial c_t/\partial c_{t-1} = \mathrm{diag}(\Gamma_f)$ can be held near 1, so that signal survives.

**Why the sweep stops at 40.** DBpedia abstracts average **46.1 real tokens**; past ~40 you stop testing longer documents and start testing longer `[PAD]` runs, and because this encoder reads `h[-1]` the state is taken after a long tail of zero-embedding steps. That would measure the provided encoder, not recurrence. (Fixing it is an optional extension: index the last *real* token, or use `pack_padded_sequence`.)

---

## 🔬 E9 — Map the LSH speed/recall trade-off

Brute-force retrieval compares every query against all 20,000 gallery vectors. At 100 million that is hopeless. **Random-hyperplane LSH** (**Lesson 3**, eq. 3) gives each vector $K$ bits — which side of each random hyperplane it falls on — and only compares vectors sharing a bucket. Two vectors get the same bit unless a hyperplane falls between them, which happens with probability $\theta_{ab}/\pi$, so **collision probability is a monotone function of cosine similarity**.

Two knobs: $K$ (bits per table — more bits = finer buckets = fewer candidates) and $L$ (independent tables — more tables = more chances to collide = better recall). The theory says they pull in **opposite directions**. Map that.

**But recall@10 only asks whether LSH found the same *vectors* brute force would. The question that actually matters is whether it found the same *topics*.** Check both.

**Design and run the experiment:**

1. Sweep `K ∈ [8, 12, 16]` × `L ∈ [1, 3, 5]` — for each, `build_lsh(gallery_lstm, K=K, L=L)` then `lsh_eval(...)` — and store `lsh_sweep[(K, L)] = {"recall": …, "candidates": …}`. Scatter recall against candidates scanned.
2. **Does the shortcut cost you topics?** For a few configurations, retrieve with `lsh_retrieve(...)` and score the result with `topic_precision(...)`. Compare against the brute-force precision@10 you already know. Store it as `"p10"` in the same dict.
3. **Look at the words.** Print one query and the neighbours LSH actually returned, flagging each `SAME`/`DIFF` topic, so you can see what a bucket lookup is really giving you.
4. Set `chosen_KL` to your preferred operating point and defend it below.

**Then answer below:** which knob buys what, and what does it cost? Is there a configuration that is **strictly dominated** (worse on *both* axes than another)? And — the point of step 2 — how much topic quality did you actually give up for that speed-up?

> Everything you need: `build_lsh`, `lsh_eval`, `lsh_retrieve`, `topic_precision`, `gallery_lstm`, `query_lstm`.

In [ ]:
# TODO: 1) sweep K in [8,12,16] x L in [1,3,5]; record recall@10 and avg candidate-set size
# TODO: 2) for a few configs also record p10 via lsh_retrieve + topic_precision
# TODO: 3) scatter recall vs candidates; print one query's LSH neighbours as TEXT (SAME/DIFF)
# TODO: 4) set `chosen_KL` to your operating point
# HINT: tables = build_lsh(gallery_lstm, K=K, L=L);  r, c = lsh_eval(tables, query_lstm, gallery_lstm)
# HINT: topk, avg = lsh_retrieve(tables, query_lstm, gallery_lstm)
# HINT: topic_precision(topk, query_labels, gallery_labels)

# the brute-force reference: what a full 20,000-vector scan per query buys you
qn = query_lstm / query_lstm.norm(dim=1, keepdim=True).clamp_min(1e-8)
gn = gallery_lstm / gallery_lstm.norm(dim=1, keepdim=True).clamp_min(1e-8)
brute_topk = (qn @ gn.T).topk(10, dim=1).indices
brute_p10 = topic_precision(brute_topk, query_labels, gallery_labels)
print(f"brute force: p@10 = {brute_p10:.3f}  (20,000 comparisons per query)\n")

lsh_sweep = {}
for K in [8, 12, 16]:
    for L in [1, 3, 5]:
        tables = build_lsh(gallery_lstm, K=K, L=L)
        r, c = lsh_eval(tables, query_lstm, gallery_lstm)
        topk, _ = lsh_retrieve(tables, query_lstm, gallery_lstm)
        p10 = topic_precision(topk, query_labels, gallery_labels)
        lsh_sweep[(K, L)] = {"recall": r, "candidates": c, "p10": p10}
        print(f"K={K:2d} L={L}:  recall@10 = {r:.3f}   p@10 = {p10:.3f} ({p10-brute_p10:+.3f})   "
              f"{c:6.1f} candidates ({len(gallery_lstm)/max(c,1):4.1f}x smaller)")

plt.figure(figsize=(7, 4.5))
for (K, L), r in lsh_sweep.items():
    plt.scatter(r["candidates"], r["recall"], s=60)
    plt.annotate(f"K={K},L={L}", (r["candidates"], r["recall"]),
                 fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xlabel("average candidates scanned (of 20,000)"); plt.ylabel("recall@10 vs brute force")
plt.title("The LSH trade-off: recall bought with candidates")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

chosen_KL = (16, 5)   # reasoning goes in the conclusion below
print(f"\nchosen operating point: K={chosen_KL[0]}, L={chosen_KL[1]} → "
      f"recall {lsh_sweep[chosen_KL]['recall']:.3f}, p@10 {lsh_sweep[chosen_KL]['p10']:.3f} at "
      f"{lsh_sweep[chosen_KL]['candidates']:.0f} candidates")

# --- what a bucket lookup actually returns, in words ---
tables = build_lsh(gallery_lstm, K=chosen_KL[0], L=chosen_KL[1])
topk, _ = lsh_retrieve(tables, query_lstm, gallery_lstm)
for qi in [0, 1]:
    print(f"\nQUERY [{CLASS_NAMES[query_labels[qi]]}] {query_texts[qi][:90]}")
    for j in topk[qi][:3].tolist():
        if j < 0:
            continue
        flag = "SAME" if gallery_labels[j] == query_labels[qi] else "DIFF"
        print(f"   {flag} [{CLASS_NAMES[gallery_labels[j]]:8}] {gallery_texts[j][:85]}")

In [ ]:
# Light check on YOUR experiment (structure only)
try:
    assert len(lsh_sweep) >= 6, f"sweep at least 6 (K, L) combinations, got {len(lsh_sweep)}"
    assert all({"recall", "candidates", "p10"} <= set(v) for v in lsh_sweep.values()), \
        "each entry needs 'recall', 'candidates' and 'p10'"
    assert all(0.0 <= v["recall"] <= 1.0 for v in lsh_sweep.values()), "recall must be a fraction"
    assert all(0 < v["candidates"] <= 20000 for v in lsh_sweep.values())
    assert chosen_KL in lsh_sweep, "chosen_KL must be one of the points you actually measured"
    Ks = {K for K, _ in lsh_sweep}
    Ls = {L for _, L in lsh_sweep}
    assert len(Ks) >= 2 and len(Ls) >= 2, "vary BOTH K and L — one knob alone is not a trade-off map"
    best = max(lsh_sweep, key=lambda kl: lsh_sweep[kl]["recall"])
    cheap = min(lsh_sweep, key=lambda kl: lsh_sweep[kl]["candidates"])
    print("✓ E9 experiment complete")
    print(f"   {len(lsh_sweep)} configs | K ∈ {sorted(Ks)} | L ∈ {sorted(Ls)}")
    print(f"   best recall  K={best[0]},L={best[1]}: {lsh_sweep[best]['recall']:.3f} "
          f"@ {lsh_sweep[best]['candidates']:.0f} candidates")
    print(f"   cheapest     K={cheap[0]},L={cheap[1]}: {lsh_sweep[cheap]['recall']:.3f} "
          f"@ {lsh_sweep[cheap]['candidates']:.0f} candidates")
    print(f"   your choice  K={chosen_KL[0]},L={chosen_KL[1]}: "
          f"recall {lsh_sweep[chosen_KL]['recall']:.3f}, p@10 {lsh_sweep[chosen_KL]['p10']:.3f} "
          f"@ {lsh_sweep[chosen_KL]['candidates']:.0f} candidates")
except NameError:
    print("✗ E9 not attempted yet — write the experiment in the cell above, then re-run.")
except Exception as e:
    print(f"✗ E9 check failed — {type(e).__name__}: {e}")


### ✍️ E9 conclusion


> TODO: your conclusion here (3–4 sentences)

**Model answer.**

**The two knobs pull in opposite directions, as advertised.** Raising $K$ makes buckets finer: at $L{=}1$, $K = 8 \to 16$ cut candidates **1132 → 529** but dropped recall **0.845 → 0.703**. Raising $L$ buys recall with work: at $K{=}12$, $L = 1 \to 5$ lifted recall **0.774 → 0.977** while candidates grew **743 → 1658**.

**Two configurations are strictly dominated** — worse on *both* axes than another point, so nothing could justify them. $K{=}8,L{=}1$ (recall 0.845 @ 1132) loses to $K{=}16,L{=}3$ (**0.923** @ **993**); and $K{=}12,L{=}3$ (0.959 @ 1352) loses to $K{=}16,L{=}5$ (**0.967** @ **1262**). On this index, *more bits with more tables* beats *fewer bits with fewer tables*, so I chose $K{=}16,L{=}5$.

**The headline is how little topic quality the shortcut costs.** Brute force scans all 20,000 vectors for **p@10 = 0.966**. My operating point scans **1,262** — **15.8× smaller** — for **p@10 = 0.964**, a loss of **0.002**. Even the most aggressive setting in the sweep ($K{=}16,L{=}1$, only 529 candidates, **37.8×** smaller) still returns **0.929**.

**Recall and precision@10 are not equally fragile, and that is the lesson.** recall@10 ranges **0.703 → 0.988** across the sweep while p@10 moves only **0.929 → 0.966**. Missing an exact nearest neighbour barely hurts, because whatever replaces it is usually *another document of the same class* — the topic is redundantly encoded across many nearby vectors. LSH doesn't need to find the same *vectors* brute force would; it only needs to land in the right neighbourhood.

---

## Wrap-up — what you actually did

You were handed a working document-retrieval sandbox and used it to produce three findings that were **not** in the notebook:

- **E7** — you measured how much of the pipeline's quality actually depends on word order, and found the order-blind baseline is hard to beat for a reason.
- **E8** — you turned Lesson 2's exponential-decay claim into a falsifiable prediction about sequence length, and had to decide what to do when the data stopped cooperating.
- **E9** — you mapped a two-knob trade-off, had to *choose*, and measured what the speed-up actually cost in topic quality.

**Concept check (discuss aloud):**

1. E7 found shuffling costs the LSTM 6.5% of its retrieval quality — real, but small next to the encoder's own `cos_self` move (0.9526). Why is retrieval so much less sensitive to the shuffle than the embedding itself is?
2. E8 found the RNN collapses to chance by length 40 while the LSTM stays flat. Does that contradict E7's finding that order "only" costs 6.5%? What does each experiment actually measure?
3. E9's cheapest LSH setting still returned same-topic neighbours almost as well as scanning all 20,000. Why does missing an *exact* nearest neighbour cost so little here?

**Before you're done:** *Kernel → Restart & Run All*, and confirm every `✓` printed — that's your signal all three experiments actually ran.

### Optional extensions

- **Fix E8's padding bug** — index the last *real* token instead of `h[-1]` and re-run the length sweep.
- **Freeze vs. fine-tune** — set `enc.embedding.weight.requires_grad = False` and re-run E7.
- **Retrain on shuffled text** — E7 asked whether a trained encoder *uses* order; ask instead whether one can *learn* without it.
- **Attention pooling** — replace "take $h_T$" with $z = \sum_j \alpha_j h_j$ (**Lesson 2**, eq. 15) — a direct bridge to Module 2.

**That closes Module 1's in-class labs.** Next up: **Lesson 4**, where these same pieces are assembled into a system that *generates* — a sequence-to-sequence translator.

**Data & tools:** DBpedia-14 (Zhang, Zhao & LeCun, 2015) via HF `datasets`; GloVe `glove-wiki-gigaword-50` (Pennington, Socher & Manning, 2014) via `gensim`; PyTorch; LSH: Charikar, 2002.